# Cuaderno 03: Analisis Estadistico de la Variable Objetivo y Seleccion de Caracteristicas

## 1. Planteamiento del Problema: Asimetria en Metricas de Redes Sociales
La variable objetivo del sistema cuantifica la tasa de interaccion relativa por visualizacion:

$$\text{engagement\_rate} = \frac{\text{likes} + \text{comments} + \text{shares}}{\text{views}}$$

En plataformas de video corto, esta proporcion presenta dos riesgos estadisticos criticos:
1. **Inestabilidad por denominador pequeno:** Publicaciones con muy pocas reproducciones (`views < MIN_VIEWS = 100`) generan tasas extremas artificiales por tamano muestral insuficiente.
2. **Asimetria positiva extrema (cola pesada derecha):** Pocos contenidos virales concentran millones de vistas e interacciones mientras la mayoria permanece en rangos bajos, violando supuestos de homocedasticidad.

Para mitigar ambos fenomenos se aplica el filtro determinista `filtrar_vistas_minimas()` y la transformacion logaritmica `ln(1 + x)` (`numpy.log1p`), que preserva la definicion en cero (`ln(1 + 0) = 0`) y comprime la cola superior.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

raiz = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from config import settings
from src.ingesta.cargador_kaggle import cargar_dataset_tiktok
from src.modelos.evaluador_metricas import cargar_datos_entrenamiento
from src.preprocesamiento import calcular_engagement_rate, filtrar_vistas_minimas

df_modelo_crudo, fuente_activa = cargar_datos_entrenamiento()
df_modelo_filtrado = filtrar_vistas_minimas(df_modelo_crudo, min_views=settings.MIN_VIEWS)
df_modelo = calcular_engagement_rate(df_modelo_filtrado)

df_tiktok_crudo = cargar_dataset_tiktok()
df_tiktok_filtrado = filtrar_vistas_minimas(df_tiktok_crudo, min_views=settings.MIN_VIEWS)
df_tiktok = calcular_engagement_rate(df_tiktok_filtrado)

df_ig_crudo = pd.read_csv(settings.DATOS_CRUDOS_DIR / 'instagram' / 'instagram_reels_clean.csv')
# instagram_reels_clean.csv solo reporta likes y comments
df_ig_crudo['shares'] = 0.0
df_ig_filtrado = filtrar_vistas_minimas(df_ig_crudo, min_views=settings.MIN_VIEWS)
df_ig = calcular_engagement_rate(df_ig_filtrado)

resumen_filtro = pd.DataFrame([
    {
        'conjunto': f'Modelado ({fuente_activa})',
        'n_bruto': len(df_modelo_crudo),
        'n_retenido': len(df_modelo),
        'descartados_pct': 100.0 * (len(df_modelo_crudo) - len(df_modelo)) / len(df_modelo_crudo),
        'views_min': df_modelo_crudo['views'].min(),
        'views_mediana': df_modelo['views'].median(),
        'views_media': df_modelo['views'].mean(),
        'views_max': df_modelo['views'].max(),
    },
    {
        'conjunto': 'TikTok crudo (Kaggle)',
        'n_bruto': len(df_tiktok_crudo),
        'n_retenido': len(df_tiktok),
        'descartados_pct': 100.0 * (len(df_tiktok_crudo) - len(df_tiktok)) / len(df_tiktok_crudo),
        'views_min': df_tiktok_crudo['views'].min(),
        'views_mediana': df_tiktok['views'].median(),
        'views_media': df_tiktok['views'].mean(),
        'views_max': df_tiktok['views'].max(),
    },
    {
        'conjunto': 'Instagram Reels (Kaggle)',
        'n_bruto': len(df_ig_crudo),
        'n_retenido': len(df_ig),
        'descartados_pct': 100.0 * (len(df_ig_crudo) - len(df_ig)) / len(df_ig_crudo),
        'views_min': df_ig_crudo['views'].min(),
        'views_mediana': df_ig['views'].median(),
        'views_media': df_ig['views'].mean(),
        'views_max': df_ig['views'].max(),
    },
])
print(f'Fuente de modelado activa: {fuente_activa} | Umbral MIN_VIEWS: {settings.MIN_VIEWS}')
print(resumen_filtro.to_string(index=False))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=('Cola pesada en vistas (conjunto de modelado)', 'Engagement rate crudo por conjunto'),
)
fig.add_trace(
    go.Histogram(x=df_modelo['views'], nbinsx=40, name=fuente_activa, marker_color='#6366f1'),
    row=1,
    col=1,
)
fig.add_trace(
    go.Box(y=df_modelo['engagement_rate'], name=fuente_activa, marker_color='#6366f1'),
    row=1,
    col=2,
)
fig.add_trace(
    go.Box(y=df_tiktok['engagement_rate'], name='TikTok crudo', marker_color='#22c55e'),
    row=1,
    col=2,
)
fig.add_trace(
    go.Box(y=df_ig['engagement_rate'], name='Instagram Reels', marker_color='#f59e0b'),
    row=1,
    col=2,
)
fig.update_layout(
    title='Auditoria de alcance (views) y dispersion inicial del engagement rate',
    height=420,
    showlegend=False,
)
fig.update_xaxes(title_text='Reproducciones (views)', row=1, col=1)
fig.update_yaxes(title_text='Frecuencia', row=1, col=1)
fig.update_yaxes(title_text='Engagement rate crudo', row=1, col=2)
fig.show()


Fuente de modelado activa: respaldo_tiktok_csv | Umbral MIN_VIEWS: 100
                      conjunto  n_bruto  n_retenido  descartados_pct  views_min  views_mediana  views_media   views_max
Modelado (respaldo_tiktok_csv)      800         800              0.0      503.0        57793.5 3.339567e+05   2493911.0
         TikTok crudo (Kaggle)      299         299              0.0     8689.0      5500000.0 1.711589e+07 222300000.0
      Instagram Reels (Kaggle)       88          88              0.0     8919.0       217533.0 4.655332e+05   3198411.0


<plotly.graph_objs.Figure>

### Analisis del Filtro `MIN_VIEWS = 100` y Sesgo de Cola Pesada en Alcance
- **Retencion de registros:** Al no existir `datos/procesados/matriz_acustica_cruda.parquet`, `cargar_datos_entrenamiento()` activa el respaldo oficial `respaldo_tiktok_csv` ($n = 800$). Tanto en el respaldo (minimo `503` vistas) como en TikTok crudo ($n = 299$, minimo `8,689` vistas) e Instagram Reels ($n = 88$, minimo `8,919` vistas), el **0.0 %** de registros cae por debajo de `MIN_VIEWS = 100`, reteniendose el 100 % de las muestras disponibles. No obstante, conservar este filtro como compuerta obligatoria previene ruido de tasa ante futuras ingestas locales con exposicion incipiente.
- **Cola pesada en visualizaciones:** En `respaldo_tiktok_csv`, la media de `views` (`333,956.7`) multiplica por **5.78x** a la mediana (`57,793.5`) y alcanza un maximo de `2,493,911`. En TikTok crudo la brecha es aun mas severa: mediana de `5,500,000` frente a un maximo de `222,300,000` vistas. Asimismo, los diagramas de caja muestran valores atipicos extremos en el `engagement_rate` crudo de Instagram Reels (maximo superior a `0.45`) y TikTok.


In [2]:
from scipy import stats
from src.preprocesamiento import transformar_target_log

df_modelo_log = transformar_target_log(df_modelo)
df_tiktok_log = transformar_target_log(df_tiktok)
df_ig_log = transformar_target_log(df_ig)

filas_stats = []
for etiqueta, df_eval in [
    (f'Modelado ({fuente_activa})', df_modelo_log),
    ('TikTok crudo (Kaggle)', df_tiktok_log),
    ('Instagram Reels (Kaggle)', df_ig_log),
]:
    x_raw = df_eval[settings.COLUMNA_TARGET].to_numpy(dtype=float)
    x_log = df_eval[settings.COLUMNA_TARGET_LOG].to_numpy(dtype=float)
    w_raw, p_raw = stats.shapiro(x_raw)
    w_log, p_log = stats.shapiro(x_log)
    filas_stats.append(
        {
            'conjunto': etiqueta,
            'n': len(df_eval),
            'skew_raw': float(stats.skew(x_raw)),
            'skew_log': float(stats.skew(x_log)),
            'kurt_raw': float(stats.kurtosis(x_raw)),
            'kurt_log': float(stats.kurtosis(x_log)),
            'shapiro_W_raw': float(w_raw),
            'shapiro_p_raw': float(p_raw),
            'shapiro_W_log': float(w_log),
            'shapiro_p_log': float(p_log),
        }
    )

df_normalidad = pd.DataFrame(filas_stats)
print('Comparacion de asimetria, curtosis y prueba de Shapiro-Wilk (x vs ln(1+x)):')
print(df_normalidad.round(4).to_string(index=False))

fig_norm = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f'Modelado ({fuente_activa}): x vs ln(1+x)',
        'TikTok crudo real: x vs ln(1+x)',
    ),
)
fig_norm.add_trace(
    go.Histogram(x=df_modelo_log[settings.COLUMNA_TARGET], nbinsx=30, name='x crudo', opacity=0.6, marker_color='#ef4444'),
    row=1,
    col=1,
)
fig_norm.add_trace(
    go.Histogram(x=df_modelo_log[settings.COLUMNA_TARGET_LOG], nbinsx=30, name='ln(1+x)', opacity=0.6, marker_color='#22c55e'),
    row=1,
    col=1,
)
fig_norm.add_trace(
    go.Histogram(x=df_tiktok_log[settings.COLUMNA_TARGET], nbinsx=30, name='x crudo TikTok', opacity=0.6, marker_color='#ef4444', showlegend=False),
    row=1,
    col=2,
)
fig_norm.add_trace(
    go.Histogram(x=df_tiktok_log[settings.COLUMNA_TARGET_LOG], nbinsx=30, name='ln(1+x) TikTok', opacity=0.6, marker_color='#22c55e', showlegend=False),
    row=1,
    col=2,
)
fig_norm.update_layout(
    barmode='overlay',
    title='Efecto de la transformacion ln(1 + x) sobre la distribucion del engagement rate',
    height=420,
)
fig_norm.update_xaxes(title_text='Engagement rate / ln(1 + engagement)', row=1, col=1)
fig_norm.update_xaxes(title_text='Engagement rate / ln(1 + engagement)', row=1, col=2)
fig_norm.update_yaxes(title_text='Frecuencia', row=1, col=1)
fig_norm.show()


Comparacion de asimetria, curtosis y prueba de Shapiro-Wilk (x vs ln(1+x)):
                      conjunto   n  skew_raw  skew_log  kurt_raw  kurt_log  shapiro_W_raw  shapiro_p_raw  shapiro_W_log  shapiro_p_log
Modelado (respaldo_tiktok_csv) 800   -0.0877   -0.1303    0.0278    0.0580         0.9982         0.5855         0.9977         0.3485
         TikTok crudo (Kaggle) 299    0.9547    0.8053    0.4439    0.0680         0.9258         0.0000         0.9414         0.0000
      Instagram Reels (Kaggle)  88    5.3932    4.4636   37.5223   27.8666         0.5388         0.0000         0.6257         0.0000


<plotly.graph_objs.Figure>

### Justificacion Matematica de `ln(1 + x)` y Estabilizacion de Varianza
Los contrastes estadisticos verifican dos comportamientos complementarios segun el origen de los datos:
1. **Reduccion de asimetria y curtosis en redes sociales reales:** En `TikTok crudo (Kaggle)` ($n = 299$), `ln(1 + x)` reduce la asimetria positiva de `0.9547` a `0.8053`, comprime el exceso de curtosis de `0.4439` a `0.0680` (casi mesocurtica) y eleva el estadistico de Shapiro-Wilk $W$ de `0.9258` a `0.9414`. En `Instagram Reels` ($n = 88$), la asimetria desciende de `5.3932` a `4.4636`, la curtosis cae de `37.5223` a `27.8666` y $W$ sube de `0.5388` a `0.6257`. Esto demuestra matematicamente que el mapeo logaritmico contrae la cola derecha extrema y atenua la heterocedasticidad.
2. **Preservacion de normalidad en el respaldo tabular (`respaldo_tiktok_csv`, $n = 800$):** Dado que el respaldo sintetiza la tasa latente mediante una funcion sigmoide acotada, la distribucion original ya es simetrica (`skew = -0.0877`, $W = 0.9982$, $p = 0.5855$). Aplicar `ln(1 + x)` mantiene la cercania gaussiana (`skew = -0.1303`, `kurtosis = 0.0580`, $W = 0.9977$, $p = 0.3485$), garantizando compatibilidad de contrato tanto con el respaldo actual como con futuras matrices de audio real con cola pesada.


In [3]:
correlaciones = []
y_log = df_modelo_log[settings.COLUMNA_TARGET_LOG]
for col in settings.COLUMNAS_ACUSTICAS:
    rho, p_val = stats.spearmanr(df_modelo_log[col], y_log)
    correlaciones.append(
        {
            'variable': col,
            'spearman_rho': float(rho),
            'abs_rho': float(abs(rho)),
            'p_valor': float(p_val),
        }
    )

df_corr = pd.DataFrame(correlaciones).sort_values('abs_rho', ascending=False).reset_index(drop=True)
print('Ranking de correlacion de Spearman de las 28 variables acusticas con engagement_rate_log:')
print(df_corr[['variable', 'spearman_rho', 'p_valor']].to_string(index=False))

df_corr_plot = df_corr.sort_values('spearman_rho', ascending=True)
colores = ['#22c55e' if r >= 0 else '#ef4444' for r in df_corr_plot['spearman_rho']]
fig_corr = go.Figure(
    go.Bar(
        x=df_corr_plot['spearman_rho'],
        y=df_corr_plot['variable'],
        orientation='h',
        marker_color=colores,
        hovertemplate='%{y}: rho=%{x:+.4f}<extra></extra>',
    )
)
fig_corr.update_layout(
    title='Correlacion de Spearman entre las 28 variables acusticas y engagement_rate_log',
    xaxis_title='Coeficiente rho de Spearman',
    yaxis_title='Descriptor acustico',
    height=720,
    margin=dict(l=170, r=30, t=55, b=45),
)
fig_corr.show()


Ranking de correlacion de Spearman de las 28 variables acusticas con engagement_rate_log:
           variable  spearman_rho      p_valor
onset_strength_mean      0.530224 3.362767e-59
initial_silence_sec     -0.504463 7.193449e-53
            rms_std      0.502101 2.575242e-52
  spectral_centroid      0.492671 3.791710e-50
 spectral_bandwidth      0.481051 1.441990e-47
             mfcc_1      0.477114 1.025363e-46
   spectral_rolloff      0.476019 1.761557e-46
      dynamic_range      0.475420 2.366494e-46
 onset_strength_std      0.469326 4.611886e-45
           rms_mean      0.458592 7.468871e-43
      silence_ratio     -0.392873 6.350918e-31
   chroma_stft_mean      0.268068 1.240214e-14
    beat_regularity      0.238452 8.334248e-12
             mfcc_2      0.211394 1.559071e-09
 zero_crossing_rate      0.182014 2.178344e-07
             mfcc_3      0.173160 8.337287e-07
             mfcc_6      0.158322 6.813517e-06
  spectral_flatness      0.115047 1.115068e-03
             mfcc

<plotly.graph_objs.Figure>

## 2. Conclusion Ejecutiva
1. **Impulsores fisicos de asociacion monotona:** Entre las 28 caracteristicas acusticas evaluadas en `respaldo_tiktok_csv` ($n = 800$), diez superan $|\rho| > 0.45$ con significancia estadistica extrema ($p < 10^{-42}$). Destacan positivamente la fuerza media de ataques (`onset_strength_mean`, $\rho = +0.5302$), la desviacion de energia (`rms_std`, $\rho = +0.5021$), el centroide espectral (`spectral_centroid`, $\rho = +0.4927$), el ancho de banda (`spectral_bandwidth`, $\rho = +0.4811$) y el primer coeficiente cepstral (`mfcc_1`, $\rho = +0.4771$). En sentido opuesto, el silencio inicial (`initial_silence_sec`, $\rho = -0.5045$) y la proporcion global de silencio (`silence_ratio`, $\rho = -0.3929$) penalizan fuertemente la interaccion.
2. **Colinealidad espectral y limite de la correlacion monotona:** Los descriptores energeticos y espectrales (`rms_mean`, `rms_std`, `dynamic_range`, `spectral_centroid`, `spectral_rolloff`) forman bloques altamente colineales, lo que exige regularizacion L1/L2 en regresores lineales. Mas importante aun, `tempo_bpm` exhibe una correlacion de Spearman casi nula ($\rho = +0.0425$, $p = 0.2302$) porque su relacion con el engagement es **no monotona** (penalizacion por distancia respecto a una zona optima en torno a 122-126 BPM). Esta no linealidad justifica evaluar modelos basados en arboles (`RandomForest`, `XGBoost`) y redes neuronales frente a `Ridge` y `ElasticNet` en el Cuaderno 04.
